# Retrieval study on 25 questions: bi-encoder alone vs bi-encoder + cross-encoder

Same corpus and chunks as the main notebook (D&D 5e SRD 5.1, 200-word chunks, 40-word overlap). For each of two embedders
the top-10 candidates are re-ranked by `cross-encoder/ms-marco-MiniLM-L-6-v2`, and the rank of the **gold chunk** (the
chunk that contains the passage answering the question) is recorded before and after.

This notebook only measures retrieval. It saves the retrieved chunk ids to `retrieval.json`, which the generation
notebook (`rag_generation_qwen35_9b.ipynb`) reads, so every LLM sees exactly the same contexts.

* **Embedders:** `BAAI/bge-small-en-v1.5`, `google/embeddinggemma-2` (float32)
* **Metrics:** recall@3 (is the gold chunk in the top 3 that the LLM would see?), recall@10 (is it retrievable at all?), MRR.

In [1]:
import os, re, time, json, gc
import numpy as np, pandas as pd, faiss, pymupdf, torch
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF = "/home2/akshat.pandey/rag/srd.pdf"
OUT = "/home2/akshat.pandey/rag/outputs_25q"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda"
print("gpu:", torch.cuda.get_device_name(0))

gpu: NVIDIA GeForce RTX 2080 Ti


## 1. Chunks

Pages are two-column, so each page is read as a left half and a right half; a 40 pt strip at the top and bottom is dropped.

In [2]:
CHUNK_WORDS, OVERLAP = 200, 40
doc = pymupdf.open(PDF)
words, word_page = [], []
for pno, page in enumerate(doc, 1):
    r = page.rect
    top, bot, mid = r.y0 + 40, r.y1 - 40, (r.x0 + r.x1) / 2
    for col in (pymupdf.Rect(r.x0, top, mid, bot), pymupdf.Rect(mid, top, r.x1, bot)):
        for w in page.get_text("text", clip=col).split():
            words.append(w); word_page.append(pno)
chunks = [{"id": i, "page": word_page[s], "text": " ".join(words[s:s + CHUNK_WORDS])}
          for i, s in enumerate(range(0, len(words), CHUNK_WORDS - OVERLAP))]
print(len(chunks), "chunks")

1504 chunks


## 2. The 25 questions and their gold phrases

The first 8 were picked because they show retrieval failures; the other 17 are ordinary rules questions (including some
"neutral" ones where I expected no problem). The gold phrase is a regex for text that appears only in the passage that
answers the question. Each phrase was checked by printing the chunk it matches (see `gold_phrase_check.txt`); six loose
phrases from the first draft were tightened because they matched dozens of unrelated chunks.

In [3]:
# 25 test queries: (category, query, gold regex = phrase found only in the passage that answers it)
TESTS = [
 # --- original 8 (Q2 / Q8 gold phrases corrected) ---
 ("Keyword trap",          "What breaks my focus on a spell?",                          r"incapacitated or killed"),
 ("Negation / constraint", "Do I need to see a target to attack it?",                   r"target that you can.t see"),
 ("Exception / corner",    "Can I punish an unseen enemy that runs away from me?",      r"hostile creature that you can see moves out of your reach"),
 ("Negation / constraint", "Can I cast a spell while a silence effect is on me?",       r"can.t cast a spell with a verbal component"),
 ("Exception / corner",    "Can I shove a creature much bigger than me?",               r"push it away from you"),
 ("Keyword trap (synonym)","How many magic items can one creature bond with?",          r"no more than three magic items"),
 ("Counter-example",       "Can I rest twice in a single day to recover everything?",   r"more than one long rest in a 24"),
 ("Counter-example",       "Can a creature that is held by an enemy still walk away?",  r"grappled creature.s speed becomes 0"),
 # --- 17 new ---
 ("Exception / corner",    "Does teleporting away from an enemy provoke an attack?",    r"don.t provoke an opportunity attack when you teleport"),
 ("Negation / constraint", "Can a surprised character act in the first round?",         r"can.t move or take an action"),
 ("Neutral rule",          "What happens if a character plunges off a cliff?",          r"damage for every 10 feet it fell"),
 ("Neutral rule",          "When does a character die outright instead of falling unconscious?", r"Massive damage can kill you instantly"),
 ("Exception / corner",    "Can I cast a second leveled spell after a bonus action spell?", r"except for a cantrip with a casting time of 1 action"),
 ("Negation / constraint", "Can I concentrate on two spells at once?",                  r"on two spells at once"),
 ("Neutral rule",          "Can I recover from exhaustion by resting?",                 r"exhaustion level by 1"),
 ("Neutral rule",          "How long can a creature hold its breath?",                  r"hold its breath for a number of minutes"),
 ("Exception / corner",    "Can two clerics cast the same blessing on one ally to stack it?", r"most potent effect"),
 ("Exception / corner",    "Can I disengage and still be hit as I leave?",              r"avoid provoking an opportunity attack"),
 ("Negation / constraint", "Can I take two reactions before my next turn?",             r"can.t take another one until the start of your next turn"),
 ("Neutral rule",          "Is a paralyzed creature able to act?",                      r"paralyzed creature is incapacitated"),
 ("Negation / constraint", "Can I double my proficiency bonus twice on the same roll?", r"can.t be added to a single die roll or other number more than once"),
 ("Neutral rule",          "Can advantage and disadvantage cancel each other out?",     r"neither advantage nor disadvantage"),
 ("Exception / corner",    "Can the Dash action be used as a bonus action?",            r"only to take the Dash, Disengage, or Hide"),
 ("Neutral rule",          "Can a prone creature stand up for free?",                   r"Standing up takes more effort"),
 ("Exception / corner",    "Can a monster use a legendary action in the middle of its own turn?", r"called legendary actions"),
]

In [4]:
def gold_chunks(gold):
    g = re.compile(gold, re.I)
    return [c["id"] for c in chunks if g.search(c["text"])]

def gold_rank(order_ids, gold_ids):
    for r, cid in enumerate(order_ids, 1):
        if cid in gold_ids:
            return r
    return None

GOLD = {q: gold_chunks(g) for _, q, g in TESTS}
for n, (cat, q, g) in enumerate(TESTS, 1):
    print(f"{n:>2}. {len(GOLD[q]):>2} gold chunk(s) {GOLD[q][:4]}  {q}")
assert all(0 < len(v) <= 4 for v in GOLD.values()), "a gold phrase matches nothing or too many chunks"

 1.  1 gold chunk(s) [372]  What breaks my focus on a spell?
 2.  1 gold chunk(s) [337]  Do I need to see a target to attack it?
 3.  1 gold chunk(s) [340]  Can I punish an unseen enemy that runs away from me?
 4.  1 gold chunk(s) [369]  Can I cast a spell while a silence effect is on me?
 5.  1 gold chunk(s) [343]  Can I shove a creature much bigger than me?
 6.  1 gold chunk(s) [781]  How many magic items can one creature bond with?
 7.  1 gold chunk(s) [309]  Can I rest twice in a single day to recover everything?
 8.  1 gold chunk(s) [1373]  Can a creature that is held by an enemy still walk away?
 9.  2 gold chunk(s) [340, 341]  Does teleporting away from an enemy provoke an attack?
10.  1 gold chunk(s) [317]  Can a surprised character act in the first round?
11.  1 gold chunk(s) [300]  What happens if a character plunges off a cliff?
12.  2 gold chunk(s) [351, 352]  When does a character die outright instead of falling unconscious?
13.  1 gold chunk(s) [367]  Can I cast a second 

## 3. Stage 1 (embedder) and stage 2 (cross-encoder) for each embedder

The cross-encoder only re-orders the same ten candidates the embedder returned (asserted).

In [5]:
EMBEDDERS = {
    "bge-small": dict(name="BAAI/bge-small-en-v1.5",
                      q_prefix="Represent this sentence for searching relevant passages: ", d_prefix="", kwargs={}),
    "embeddinggemma-2": dict(name="google/embeddinggemma-2",
                      q_prefix="task: search result | query: ", d_prefix="title: none | text: ",
                      kwargs={"torch_dtype": torch.float32}),     # model card: bf16 / fp32 only, fp16 can give NaN
}
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

def timeit(fn, reps=10):
    fn(); torch.cuda.synchronize(); ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
    return 1000 * float(np.mean(ts))

retrieval, emb_info = {}, {}
for key, cfg in EMBEDDERS.items():
    model = SentenceTransformer(cfg["name"], device=DEVICE, model_kwargs=cfg["kwargs"])
    t0 = time.perf_counter()
    E = model.encode([cfg["d_prefix"] + c["text"] for c in chunks], normalize_embeddings=True, batch_size=32)
    torch.cuda.synchronize(); t_index = time.perf_counter() - t0
    assert np.isfinite(E).all(), f"{key}: NaN/inf in embeddings"
    index = faiss.IndexFlatIP(E.shape[1]); index.add(E.astype("float32"))

    def stage1(q, k=10):
        qv = model.encode([cfg["q_prefix"] + q], normalize_embeddings=True).astype("float32")
        s, ids = index.search(qv, k)
        return [int(i) for i in ids[0]]

    retrieval[key] = {}
    for cat, q, g in TESTS:
        base = stage1(q)
        sc = ce.predict([(q, chunks[i]["text"]) for i in base], batch_size=10)
        order = [base[j] for j in np.argsort(-sc)]
        assert set(order) == set(base)
        retrieval[key][q] = dict(category=cat, base_ids=base, rerank_ids=order, gold_ids=GOLD[q],
                                 gold_rank_base=gold_rank(base, GOLD[q]), gold_rank_rerank=gold_rank(order, GOLD[q]),
                                 ce={base[j]: float(sc[j]) for j in range(10)})
    fixed = {q: stage1(q) for _, q, _ in TESTS}
    emb_info[key] = dict(dim=int(E.shape[1]), index_seconds=round(t_index, 1),
        stage1_ms=round(timeit(lambda: [stage1(q) for _, q, _ in TESTS]) / len(TESTS), 1),
        stage2_ms=round(timeit(lambda: [ce.predict([(q, chunks[i]["text"]) for i in fixed[q]], batch_size=10) for _, q, _ in TESTS]) / len(TESTS), 1))
    print(key, emb_info[key])
    del model, index, E; gc.collect(); torch.cuda.empty_cache()

json.dump(dict(retrieval=retrieval, emb_info=emb_info, chunks=chunks), open(f"{OUT}/retrieval.json", "w"))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

bge-small {'dim': 384, 'index_seconds': 4.3, 'stage1_ms': 10.5, 'stage2_ms': 18.7}


Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

embeddinggemma-2 {'dim': 768, 'index_seconds': 25.2, 'stage1_ms': 56.7, 'stage2_ms': 18.6}


## 4. Metrics

* **recall@3** counts queries whose gold chunk is in the top 3 (what the LLM sees).
* **recall@10** counts queries whose gold chunk is anywhere in the 10 candidates. Re-ranking cannot change this number,
  and it is the ceiling for recall@3 after re-ranking.
* **rescued** = gold chunk moved from outside the top 3 into it; **lost** = moved from inside to outside.
* **MRR** = mean of 1/rank of the gold chunk (0 when it is not in the top 10).

In [6]:
def mrr(ranks): return float(np.mean([1 / r if r else 0 for r in ranks]))
rows = []
for key in EMBEDDERS:
    R = list(retrieval[key].values())
    rb = [x["gold_rank_base"] for x in R]; rr = [x["gold_rank_rerank"] for x in R]
    inb = [r is not None and r <= 3 for r in rb]; inr = [r is not None and r <= 3 for r in rr]
    both = [(b, a) for b, a in zip(rb, rr) if b and a]
    rows.append({"embedder": key, "n": len(R),
                 "recall@3 base": sum(inb), "recall@3 rerank": sum(inr), "recall@10": sum(r is not None for r in rb),
                 "rescued (out→in top3)": sum((not b) and a for b, a in zip(inb, inr)),
                 "lost (in→out top3)": sum(b and (not a) for b, a in zip(inb, inr)),
                 "MRR base": round(mrr(rb), 3), "MRR rerank": round(mrr(rr), 3),
                 "ranks improved": sum(a < b for b, a in both), "ranks worsened": sum(a > b for b, a in both),
                 "unchanged": sum(a == b for b, a in both)})
metrics = pd.DataFrame(rows).set_index("embedder").T
metrics.to_csv(f"{OUT}/metrics.csv")
metrics

embedder,bge-small,embeddinggemma-2
n,25.000,25.000
recall@3 base,18.000,16.000
recall@3 rerank,21.000,21.000
recall@10,22.000,23.000
rescued (out→in top3),3.000,6.000
lost (in→out top3),0.000,1.000
MRR base,0.613,0.645
MRR rerank,0.711,0.685
ranks improved,8.000,8.000
ranks worsened,3.000,3.000


## 5. Per-question rank shifts

`None` = the gold chunk is not among the ten candidates.

In [7]:
pq = pd.DataFrame([{"#": n, "category": cat, "query": q,
                    **{f"{k}: base→rerank": f"{retrieval[k][q]['gold_rank_base']} → {retrieval[k][q]['gold_rank_rerank']}" for k in EMBEDDERS}}
                   for n, (cat, q, g) in enumerate(TESTS, 1)])
pd.set_option("display.width", 250, "display.max_colwidth", 80, "display.max_rows", 100)
pq.to_csv(f"{OUT}/per_question_ranks.csv", index=False)
pq

,#,category,query,bge-small: base→rerank,embeddinggemma-2: base→rerank
0,1,Keyword trap,What breaks my focus on a spell?,5 → 2,6 → 2
1,2,Negation / constraint,Do I need to see a target to attack it?,8 → 3,5 → 2
2,3,Exception / corner,Can I punish an unseen enemy that runs away from me?,4 → 3,5 → 5
3,4,Negation / constraint,Can I cast a spell while a silence effect is on me?,3 → 1,6 → 1
4,5,Exception / corner,Can I shove a creature much bigger than me?,3 → 1,None → None
5,6,Keyword trap (synonym),How many magic items can one creature bond with?,3 → 1,1 → 1
6,7,Counter-example,Can I rest twice in a single day to recover everything?,None → None,5 → 2
7,8,Counter-example,Can a creature that is held by an enemy still walk away?,None → None,None → None
8,9,Exception / corner,Does teleporting away from an enemy provoke an attack?,1 → 1,2 → 1
9,10,Negation / constraint,Can a surprised character act in the first round?,1 → 1,1 → 1


## 6. By question type

In [8]:
by = []
for key in EMBEDDERS:
    for cat in sorted({c for c, _, _ in TESTS}):
        R = [x for x in retrieval[key].values() if x["category"] == cat]
        ib = sum(x["gold_rank_base"] is not None and x["gold_rank_base"] <= 3 for x in R)
        ir = sum(x["gold_rank_rerank"] is not None and x["gold_rank_rerank"] <= 3 for x in R)
        by.append({"embedder": key, "category": cat, "n": len(R), "recall@3 base": ib, "recall@3 rerank": ir})
by = pd.DataFrame(by); by.to_csv(f"{OUT}/by_category.csv", index=False); by

,embedder,category,n,recall@3 base,recall@3 rerank
0,bge-small,Counter-example,2,0,0
1,bge-small,Exception / corner,8,6,7
2,bge-small,Keyword trap,1,0,1
3,bge-small,Keyword trap (synonym),1,1,1
4,bge-small,Negation / constraint,6,5,6
5,bge-small,Neutral rule,7,6,6
6,embeddinggemma-2,Counter-example,2,0,1
7,embeddinggemma-2,Exception / corner,8,5,6
8,embeddinggemma-2,Keyword trap,1,0,1
9,embeddinggemma-2,Keyword trap (synonym),1,1,1


## 7. Latency

In [9]:
lat = pd.DataFrame(emb_info).T; lat.to_csv(f"{OUT}/embedder_latency.csv"); lat

,dim,index_seconds,stage1_ms,stage2_ms
bge-small,384.0,4.3,10.5,18.7
embeddinggemma-2,768.0,25.2,56.7,18.6
